# Análisis de tráfico de Joomla — Parcial 2 Comunicaciones

Ejecutar con **Run → Run All Cells**. El cuaderno:
1. Se conecta a PostgreSQL (`database:5432`) con las credenciales que inyecta docker-compose.
2. Genera tráfico hacia Joomla **a través de nginx**.
3. Lee el log de nginx desde la tabla foránea `monitoreo.nginx_access` (file_fdw).
4. Grafica códigos HTTP, IPs y analiza la reutilización de conexiones TCP (keep-alive).

## 1. Conexión a PostgreSQL (capa 7 sobre TCP 5432)

In [ ]:
import os, socket
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

url = (f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
       f"@{os.environ['PGHOST']}:{os.environ['PGPORT']}/{os.environ['PGDATABASE']}")
engine = create_engine(url)

print(pd.read_sql("SELECT version()", engine).iloc[0, 0])

## 2. Capa 3 — Resolución DNS con el DNS embebido de Docker (127.0.0.11)

In [ ]:
print(open('/etc/resolv.conf').read())
for servicio in ["nginx", "joomla", "database", "grafana"]:
    try:
        ips = sorted({a[4][0] for a in socket.getaddrinfo(servicio, None, socket.AF_INET)})
    except socket.gaierror:
        ips = ["no resuelve"]
    print(f"{servicio:10s} -> {', '.join(ips)}")

## 3. Generar tráfico a Joomla pasando por nginx

In [ ]:
import urllib.request, urllib.error
from collections import Counter

rutas = ["/", "/index.php", "/administrator/", "/no-existe"]
codigos = []
for i in range(40):
    try:
        r = urllib.request.urlopen(f"http://nginx{rutas[i % 4]}", timeout=30)
        codigos.append(r.status)
    except urllib.error.HTTPError as e:
        codigos.append(e.code)

print(Counter(codigos))

## 4. Leer el log de nginx desde PostgreSQL (`monitoreo.nginx_access`)

In [ ]:
df = pd.read_sql("SELECT * FROM monitoreo.nginx_access ORDER BY ts", engine)
print(len(df), "peticiones registradas")
df.tail(10)

## 5. Capa 7 — Gráficas de tráfico

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df["status"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color="tab:purple")
ax[0].set_title("Peticiones por código HTTP")
ax[0].set_xlabel("código")

df["client_ip"].value_counts().plot(kind="barh", ax=ax[1], color="tab:cyan")
ax[1].set_title("IPs cliente")

plt.tight_layout()
plt.show()

## 6. Capa 4 — Reutilización de conexiones TCP (keep-alive)

In [ ]:
por_conexion = df.groupby("connection_id").size()
print("Peticiones HTTP:", len(df))
print("Conexiones TCP:", len(por_conexion))
print("Peticiones por conexión (promedio):", round(por_conexion.mean(), 2))

## Conclusiones
* Las peticiones aparecen en PostgreSQL sin ningún INSERT: la tabla foránea lee el archivo `access.tsv` que nginx escribe en un volumen compartido.
* Aparecen dos IPs cliente: `172.30.10.1` (navegador del host, tras el NAT del puerto 80) y la IP de Jupyter en `frontend_net`.
* Con keep-alive, varias peticiones HTTP viajan por una misma conexión TCP.